In [1]:
import numpy as np
from Bio import Phylo
import sys
sys.path.append('../../pysimARG')
from clonal_genealogy import ClonalTree
from ClonalOrigin_ARG import ARG
from add_mutation import add_mutation
from localtree import LocalTree
from localtree_simbac import load_local_trees, get_tree_at_position
from homoplasy_index_simbac import homoplasy_index_simbac
from newick_to_tree import newick_to_tree

## Local tree selection

In [2]:
file_path = "../../data/SimBac/local_tree.nwk" # Replace with your actual file name

blocks = load_local_trees(file_path)

print(f"\nSuccessfully loaded {len(blocks)} local trees.")

# Let's inspect the first few blocks to see where they map on the genome
print("\n--- Genome Map Overview ---")
current_position = 1

for i, (span, tree) in enumerate(blocks[:5]): # Just looking at the first 5
    end_position = current_position + span - 1
    print(f"Block {i+1}: Sites {current_position:05d} to {end_position:05d} (Length: {span}bp) - Tree has {len(tree.get_terminals())} leaves")
    current_position += span

Parsing local trees...

Successfully loaded 10618 local trees.

--- Genome Map Overview ---
Block 1: Sites 00001 to 00025 (Length: 25bp) - Tree has 10 leaves
Block 2: Sites 00026 to 00060 (Length: 35bp) - Tree has 10 leaves
Block 3: Sites 00061 to 00088 (Length: 28bp) - Tree has 10 leaves
Block 4: Sites 00089 to 00119 (Length: 31bp) - Tree has 10 leaves
Block 5: Sites 00120 to 00123 (Length: 4bp) - Tree has 10 leaves


In [3]:
# mock_blocks = [
#     (25, "Tree_A"), # Covers sites 1 to 25
#     (35, "Tree_B"), # Covers sites 26 to 60
#     (28, "Tree_C")  # Covers sites 61 to 88
# ]

# Let's look for site 40
target_position = 119

tree, start, end = get_tree_at_position(blocks, target_position)

In [4]:
tree, start, end

(Tree(rooted=False, weight=1.0), 89, 119)

In [5]:
Phylo.draw_ascii(tree)

                                                       ___________________ 4
  ____________________________________________________|
 |                                                    |                ___ 2
 |                                                    |_______________|
 |                                                                    |___ 8
 |
 |                                                         _______________ 9
_|                                                        |
 |                                                        |              , 6
 |                                                 _______|   ___________|
 |                                                |       | ,|           | 7
 |                                                |       | ||
 |                                                |       |_||____________ 3
 |________________________________________________|         |
                                                  |         |_________

In [8]:
target_position = 120

tree, start, end = get_tree_at_position(blocks, target_position)
tree, start, end

(Tree(rooted=False, weight=1.0), 120, 123)

In [9]:
Phylo.draw_ascii(tree)

                                                           _______________ 4
  ________________________________________________________|
 |                                                        |             ___ 2
 |                                                        |____________|
 |                                                                     |___ 8
 |
 |                                                             ___________ 9
_|                                                            |
 |                                                            |          , 6
 |                                                      ______|  ________|
 |                                                     |      |,|        | 7
 |                                                     |      |||
 |                                                     |      |||_________ 3
 |_____________________________________________________|       |
                                                     

## Review seq homoplasy

In [11]:
np.random.seed(100)
tree = ClonalTree(n=10)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../../data/SimBac/clonal_frame.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
tree.edge = edge
tree.node_height = node_height
tree.height = np.max(node_height)
tree.length = np.sum(edge[:, 2])

rho_site = 0.02
theta_site = 0.05
L = 2000
delta = 30

                                                                   _______ 2
                                        __________________________|
                            ___________|                          |_______ 8
                           |           |
                           |           |___________________________________ 1
                           |
                        ___|                         _____________________ 6
                       |   |                       ,|
                       |   |                       ||  ___________________ 3
                       |   |                       ||_|
                       |   |_______________________|  |___________________ 9
  _____________________|                           |
 |                     |                           |         _____________ 5
 |                     |                           |________|
_|                     |                                    |_____________ 7
 |                  

In [12]:
ARG_sim = ARG(tree, rho_site, L, delta, L, "seq")
node_site = add_mutation(ARG_sim, theta_site)

In [13]:
n_leaf = ARG_sim.n
n_site = ARG_sim.node_mat.shape[1]
m_vec = np.zeros(n_site, dtype=int)
s_vec = np.zeros(n_site, dtype=int)

In [14]:
n_leaf, n_site, m_vec, s_vec

(10,
 2000,
 array([0, 0, 0, ..., 0, 0, 0], shape=(2000,)),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2000,)))

In [15]:
arg_site = LocalTree(ARG_sim, 0)
node_vec = arg_site.node_index.astype(int)
node_vec

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  13,  15,
        17,  18,  20,  22,  24,  26,  28,  30,  32,  34,  36,  37,  39,
        41,  43,  45,  46,  48,  49,  51,  53,  54,  55,  56,  58,  60,
        62,  63,  65,  67,  69,  71,  73,  75,  76,  77,  78,  80,  82,
        84,  85,  86,  87,  88,  90,  91,  92,  93,  95,  97,  99, 100,
       102, 103, 105, 106, 107, 108, 109, 110, 112, 114, 116, 117, 119,
       120, 122, 123, 125, 126, 127, 128, 130, 131, 132, 134, 135, 137,
       139, 140, 141, 142, 144, 146, 147, 148, 150, 151, 152, 154, 155,
       157, 159, 161, 163, 165, 167, 168, 169, 171, 173, 175, 176, 177,
       178, 180, 182, 184, 186, 188, 189, 190, 191, 193, 195, 197, 199,
       200, 202, 204, 206, 207, 209, 210, 212, 213, 214, 215, 216, 218,
       219, 221, 223, 224, 226, 227, 229, 230, 231, 232, 233, 234, 236,
       238, 240, 242, 243, 244, 245, 247, 248, 249, 251, 252, 253, 255,
       257, 259])

In [16]:
arg_site.node_index

array([  1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,  11.,
        13.,  15.,  17.,  18.,  20.,  22.,  24.,  26.,  28.,  30.,  32.,
        34.,  36.,  37.,  39.,  41.,  43.,  45.,  46.,  48.,  49.,  51.,
        53.,  54.,  55.,  56.,  58.,  60.,  62.,  63.,  65.,  67.,  69.,
        71.,  73.,  75.,  76.,  77.,  78.,  80.,  82.,  84.,  85.,  86.,
        87.,  88.,  90.,  91.,  92.,  93.,  95.,  97.,  99., 100., 102.,
       103., 105., 106., 107., 108., 109., 110., 112., 114., 116., 117.,
       119., 120., 122., 123., 125., 126., 127., 128., 130., 131., 132.,
       134., 135., 137., 139., 140., 141., 142., 144., 146., 147., 148.,
       150., 151., 152., 154., 155., 157., 159., 161., 163., 165., 167.,
       168., 169., 171., 173., 175., 176., 177., 178., 180., 182., 184.,
       186., 188., 189., 190., 191., 193., 195., 197., 199., 200., 202.,
       204., 206., 207., 209., 210., 212., 213., 214., 215., 216., 218.,
       219., 221., 223., 224., 226., 227., 229., 23

In [17]:
node_site_vec = node_site[node_vec - 1, 0]
node_site_vec

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [ ]:
for site_loc in range(n_site):
    # Get local tree at this site
    arg_site = LocalTree(ARG_sim, site_loc)
    node_vec = arg_site.node_index.astype(int)
    node_site_vec = node_site[node_vec - 1, site_loc]  # Convert to 0-indexed

    # Compute minimum possible changes
    leaf_states = node_site_vec[:n_leaf]
    if np.any(leaf_states) and not np.all(leaf_states):
        m_vec[site_loc] = 1

    # Compute actual changes using Fitch algorithm
    s_site = 0
    site_dict = {}

    # Initialize leaf nodes
    site_dict = {node: {int(state)} for node, state in zip(node_vec[:n_leaf], leaf_states)}

    # Process internal nodes
    for i in range(n_leaf, len(node_vec)):
        parent_node = node_vec[i]
        node_indices = np.where(arg_site.edge[:, 0] == parent_node)[0]
        if len(node_indices) == 2:
            # Coalescent structure (two children)
            children_node = arg_site.edge[node_indices, 1].astype(int)
            child_1_states = site_dict[children_node[0]]
            child_2_states = site_dict[children_node[1]]

            intersec = child_1_states & child_2_states
            if len(intersec) == 0:
                # Intersection is empty -> mutation occurred
                site_dict[parent_node] = child_1_states | child_2_states
                s_site += 1
            else:
                # Intersection is not empty -> no mutation
                site_dict[parent_node] = intersec

        elif len(node_indices) == 1:
            # Recombination structure (single child)
            children_node = arg_site.edge[node_indices, 1].astype(int)
            site_dict[parent_node] = site_dict[children_node[0]]

    s_vec[site_loc] = s_site

# Calculate homoplasy index
m_sum = np.sum(m_vec)
s_sum = np.sum(s_vec)

In [20]:
i, leaf_states

(164,
 array([False,  True, False, False, False, False, False, False, False,
        False]))

In [21]:
# Compute actual changes using Fitch algorithm
s_site = 0
site_dict = {}

# Initialize leaf nodes
site_dict = {node: {int(state)} for node, state in zip(node_vec[:n_leaf], leaf_states)}

# Process internal nodes
for i in range(n_leaf, len(node_vec)):
    parent_node = node_vec[i]
    node_indices = np.where(arg_site.edge[:, 0] == parent_node)[0]
    if len(node_indices) == 2:
        # Coalescent structure (two children)
        children_node = arg_site.edge[node_indices, 1].astype(int)
        child_1_states = site_dict[children_node[0]]
        child_2_states = site_dict[children_node[1]]

        intersec = child_1_states & child_2_states
        if len(intersec) == 0:
            # Intersection is empty -> mutation occurred
            site_dict[parent_node] = child_1_states | child_2_states
            s_site += 1
        else:
            # Intersection is not empty -> no mutation
            site_dict[parent_node] = intersec

    elif len(node_indices) == 1:
        # Recombination structure (single child)
        children_node = arg_site.edge[node_indices, 1].astype(int)
        site_dict[parent_node] = site_dict[children_node[0]]
s_site

1

In [19]:
1 - m_sum / s_sum

np.float64(0.046875)

## Convert to SimBac local tree

In [22]:
node_site

array([[ True, False, False, ..., False, False,  True],
       [ True, False, False, ..., False, False,  True],
       [ True, False, False, ..., False, False,  True],
       ...,
       [ True, False, False, ..., False, False,  True],
       [ True, False, False, ..., False, False,  True],
       [ True, False, False, ..., False, False,  True]], shape=(283, 2000))

In [23]:
node_site = np.loadtxt("../../data/SimBac/genomes_bool.csv", delimiter=",", dtype=bool)
node_site

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]],
      shape=(10, 100000))

In [24]:
start_pos = 1
end_pos = start_pos + 10000 - 1
start_pos, end_pos

(1, 10000)

In [25]:
file_path = "../../data/SimBac/local_tree.nwk" # Replace with your actual file name

blocks = load_local_trees(file_path)

print(f"\nSuccessfully loaded {len(blocks)} local trees.")

# Let's inspect the first few blocks to see where they map on the genome
print("\n--- Genome Map Overview ---")
current_position = 1

for i, (span, tree) in enumerate(blocks[:20]): # Just looking at the first 20
    end_position = current_position + span - 1
    print(f"Block {i+1}: Sites {current_position:05d} to {end_position:05d} (Length: {span}bp) - Tree has {len(tree.get_terminals())} leaves")
    current_position += span

Parsing local trees...

Successfully loaded 10618 local trees.

--- Genome Map Overview ---
Block 1: Sites 00001 to 00025 (Length: 25bp) - Tree has 10 leaves
Block 2: Sites 00026 to 00060 (Length: 35bp) - Tree has 10 leaves
Block 3: Sites 00061 to 00088 (Length: 28bp) - Tree has 10 leaves
Block 4: Sites 00089 to 00119 (Length: 31bp) - Tree has 10 leaves
Block 5: Sites 00120 to 00123 (Length: 4bp) - Tree has 10 leaves
Block 6: Sites 00124 to 00125 (Length: 2bp) - Tree has 10 leaves
Block 7: Sites 00126 to 00136 (Length: 11bp) - Tree has 10 leaves
Block 8: Sites 00137 to 00139 (Length: 3bp) - Tree has 10 leaves
Block 9: Sites 00140 to 00149 (Length: 10bp) - Tree has 10 leaves
Block 10: Sites 00150 to 00152 (Length: 3bp) - Tree has 10 leaves
Block 11: Sites 00153 to 00153 (Length: 1bp) - Tree has 10 leaves
Block 12: Sites 00154 to 00155 (Length: 2bp) - Tree has 10 leaves
Block 13: Sites 00156 to 00165 (Length: 10bp) - Tree has 10 leaves
Block 14: Sites 00166 to 00172 (Length: 7bp) - Tree 

In [39]:
n_leaf = node_site.shape[0]
n_site = 10000
m_vec = np.zeros(n_site, dtype=int)
s_vec = np.zeros(n_site, dtype=int)
n_leaf, n_site, m_vec, s_vec

(10,
 10000,
 array([0, 0, 0, ..., 0, 0, 0], shape=(10000,)),
 array([0, 0, 0, ..., 0, 0, 0], shape=(10000,)))

Require:
- arg_site.edge
- node_vec
- leaf_states

In [27]:
local_tree, start, end = get_tree_at_position(blocks, start_pos)

In [28]:
Phylo.draw_ascii(local_tree), start, end

                                                    ______________________ 4
  _________________________________________________|
 |                                                 |                  ____ 2
 |                                                 |_________________|
 |                                                                   |____ 8
 |
_|                                                       _________________ 9
 |                                                      |
 |                                             _________|__________________ 5
 |                                            |         |
 |                                            |         |                , 6
 |                                            |         |   _____________|
 |____________________________________________|         |__|             | 7
                                              |            |
                                              |            |______________ 3
    

(None, 1, 25)

In [29]:
local_edge, local_node_height = newick_to_tree(local_tree)
local_edge, local_node_height

(array([[1.1000000e+01, 9.0000000e+00, 8.5366000e-02],
        [1.1000000e+01, 3.0000000e+00, 8.5366000e-02],
        [1.2000000e+01, 1.1000000e+01, 2.6743300e-01],
        [1.2000000e+01, 5.0000000e+00, 3.5279900e-01],
        [1.3000000e+01, 8.0000000e+00, 2.9385610e-02],
        [1.3000000e+01, 7.0000000e+00, 2.9385610e-02],
        [1.4000000e+01, 4.0000000e+00, 2.3634561e-01],
        [1.4000000e+01, 1.3000000e+01, 2.0696000e-01],
        [1.5000000e+01, 6.0000000e+00, 2.7620011e-01],
        [1.5000000e+01, 1.4000000e+01, 3.9854500e-02],
        [1.6000000e+01, 1.0000000e+01, 2.8275000e-01],
        [1.6000000e+01, 1.5000000e+01, 6.5498900e-03],
        [1.7000000e+01, 1.0000000e+00, 4.9792000e-02],
        [1.7000000e+01, 2.0000000e+00, 4.9792000e-02],
        [1.8000000e+01, 1.7000000e+01, 3.7510300e-01],
        [1.8000000e+01, 1.6000000e+01, 1.4214500e-01],
        [1.9000000e+01, 1.2000000e+01, 7.4856700e-01],
        [1.9000000e+01, 1.8000000e+01, 6.7647100e-01]]),
 array([

In [ ]:
local_pos = start_pos
local_pos

1

In [35]:
local_pos = 5
i = 4
not start <= local_pos <= end

False

In [36]:
node_vec = np.arange(1, 2*n_leaf)
node_vec

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19])

In [40]:
# Compute minimum possible changes
leaf_states = node_site[:, local_pos - 1]  # Convert to 0-indexed
if np.any(leaf_states) and not np.all(leaf_states):
    m_vec[i] = 1

In [42]:
m_vec[:10]

array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [43]:
# Compute actual changes using Fitch algorithm
s_site = 0
site_dict = {}

# Initialize leaf nodes
site_dict = {node: {int(state)} for node, state in zip(node_vec[:n_leaf], leaf_states)}
site_dict

{np.int64(1): {0},
 np.int64(2): {0},
 np.int64(3): {1},
 np.int64(4): {0},
 np.int64(5): {1},
 np.int64(6): {0},
 np.int64(7): {0},
 np.int64(8): {0},
 np.int64(9): {1},
 np.int64(10): {0}}

In [44]:
list(range(n_leaf, len(node_vec)))

[10, 11, 12, 13, 14, 15, 16, 17, 18]

In [45]:
parent_node = node_vec[10]
node_indices = np.where(local_edge[:, 0] == parent_node)[0]
parent_node, node_indices

(np.int64(11), array([0, 1]))

In [46]:
# Process internal nodes
for i in range(n_leaf, len(node_vec)):
    parent_node = node_vec[i]
    node_indices = np.where(local_edge[:, 0] == parent_node)[0]

    # Coalescent structure (two children)
    children_node = local_edge[node_indices, 1].astype(int)
    child_1_states = site_dict[children_node[0]]
    child_2_states = site_dict[children_node[1]]

    intersec = child_1_states & child_2_states
    if len(intersec) == 0:
        # Intersection is empty -> mutation occurred
        site_dict[parent_node] = child_1_states | child_2_states
        s_site += 1
    else:
        # Intersection is not empty -> no mutation
        site_dict[parent_node] = intersec

In [47]:
s_site

1

In [105]:
for i in range(n_site):
    # Get local tree at this site
    if not start <= local_pos <= end:
        local_tree, start, end = get_tree_at_position(blocks, local_pos)
        local_edge, local_node_height = newick_to_tree(local_tree)

    node_vec = np.arange(1, 2*n_leaf)

    # Compute minimum possible changes
    leaf_states = node_site[:, local_pos - 1]  # Convert to 0-indexed
    if np.any(leaf_states) and not np.all(leaf_states):
        m_vec[i] = 1

    # Compute actual changes using Fitch algorithm
    s_site = 0
    site_dict = {}

    # Initialize leaf nodes
    site_dict = {node: {int(state)} for node, state in zip(node_vec[:n_leaf], leaf_states)}

    # Process internal nodes
    for i in range(n_leaf, len(node_vec)):
        parent_node = node_vec[i]
        node_indices = np.where(local_edge[:, 0] == parent_node)[0]

        # Coalescent structure (two children)
        children_node = local_edge[node_indices, 1].astype(int)
        child_1_states = site_dict[children_node[0]]
        child_2_states = site_dict[children_node[1]]

        intersec = child_1_states & child_2_states
        if len(intersec) == 0:
            # Intersection is empty -> mutation occurred
            site_dict[parent_node] = child_1_states | child_2_states
            s_site += 1
        else:
            # Intersection is not empty -> no mutation
            site_dict[parent_node] = intersec

    s_vec[i] = s_site
    local_pos += 1

# Calculate homoplasy index
m_sum = np.sum(m_vec)
s_sum = np.sum(s_vec)

In [107]:
s_vec, s_sum

(array([0, 0, 0, ..., 0, 0, 0], shape=(10000,)), np.int64(0))

In [108]:
1 - m_sum / s_sum

C:\Users\u2008181\AppData\Local\Temp\ipykernel_7240\1426461223.py:1: RuntimeWarning: divide by zero encountered in scalar divide
  1 - m_sum / s_sum


np.float64(-inf)

## Try

In [6]:
node_site = np.loadtxt("../../data/SimBac/genomes_bool.csv", delimiter=",", dtype=bool)
node_site

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]],
      shape=(10, 100000))

In [7]:
start_pos = np.random.randint(1, 100000-2000)
start_pos

3859

In [8]:
hi_list = []
for i in range(1000):
    np.random.seed(i)
    start_pos = np.random.randint(1, 100000-2000)
    hi = homoplasy_index_simbac(blocks, node_site, start_pos, 2000)
    hi_list.append(hi)
    print(f"Iteration index: {i}")

Iteration index: 0
Iteration index: 1
Iteration index: 2
Iteration index: 3
Iteration index: 4
Iteration index: 5
Iteration index: 6
Iteration index: 7
Iteration index: 8
Iteration index: 9
Iteration index: 10
Iteration index: 11
Iteration index: 12
Iteration index: 13
Iteration index: 14
Iteration index: 15
Iteration index: 16
Iteration index: 17
Iteration index: 18
Iteration index: 19
Iteration index: 20
Iteration index: 21
Iteration index: 22
Iteration index: 23
Iteration index: 24
Iteration index: 25
Iteration index: 26
Iteration index: 27
Iteration index: 28
Iteration index: 29
Iteration index: 30
Iteration index: 31
Iteration index: 32
Iteration index: 33
Iteration index: 34
Iteration index: 35
Iteration index: 36
Iteration index: 37
Iteration index: 38
Iteration index: 39
Iteration index: 40
Iteration index: 41
Iteration index: 42
Iteration index: 43
Iteration index: 44
Iteration index: 45
Iteration index: 46
Iteration index: 47
Iteration index: 48
Iteration index: 49
Iteration 

In [9]:
hi_list

[np.float64(0.02473498233215543),
 np.float64(0.02316602316602312),
 np.float64(0.03663003663003661),
 np.float64(0.03600000000000003),
 np.float64(0.0304347826086957),
 np.float64(0.04263565891472865),
 np.float64(0.02930402930402931),
 np.float64(0.02631578947368418),
 np.float64(0.03956834532374098),
 np.float64(0.02991452991452992),
 np.float64(0.029288702928870314),
 np.float64(0.013468013468013518),
 np.float64(0.004273504273504258),
 np.float64(0.02316602316602312),
 np.float64(0.008163265306122436),
 np.float64(0.03716216216216217),
 np.float64(0.02608695652173909),
 np.float64(0.019323671497584516),
 np.float64(0.02392344497607657),
 np.float64(0.019230769230769273),
 np.float64(0.024137931034482807),
 np.float64(0.007751937984496138),
 np.float64(0.028000000000000025),
 np.float64(0.02033898305084747),
 np.float64(0.026119402985074647),
 np.float64(0.03703703703703709),
 np.float64(0.01526717557251911),
 np.float64(0.0444444444444444),
 np.float64(0.03942652329749108),
 np.fl